[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/06_predictive_validation_and_perturbation.ipynb)

# Step 06 - Predictive validation and perturbation robustness

This notebook runs the local Step 06 pipeline without Google Drive dependencies and writes auditable outputs under `outputs/predictive_validation/`.

**Scope:** reviewer-facing validation uses all accepted cells by default (`candidate_policy='best_per_cell'`) and all six canonical currents for perturbations. This is a full cell-level validation scope: all accepted cells are represented by one best accepted candidate per cell, avoiding arbitrary candidate-row caps while preventing a candidate-rich cell from dominating the perturbation audit.

**Claim scope:** Step 06 may support a predictive/perturbation robustness screen, but it does not by itself authorize final biological degeneracy wording. Later assumption-sensitivity and parameter-plausibility steps remain required.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
PROJECT_ROOT

In [ ]:
from src.step06_predictive_validation import Step06Config, run_step06_predictive_validation

config = Step06Config(
    max_candidates=None,
    candidate_policy="best_per_cell",
    time_points=50,
    perturbation_current_na=None,
    write_outputs=True,
)
result = run_step06_predictive_validation(PROJECT_ROOT, config)
result["analysis_summary"]

## Accepted ensemble and mechanism inventory

In [ ]:
heldout = result["heldout_current_errors"]
heldout[["file_id", "region", "condition", "candidate_id", "mechanism_cluster"]].drop_duplicates().head(10)

## Held-out-current error table

In [ ]:
heldout.head(12)

## Prediction interval table and figure

In [ ]:
intervals = result["prediction_intervals"]
intervals.head(12)

In [ ]:
plot_df = intervals[intervals["feature"] == "peak_depolarization_mV"].copy()
fig, ax = plt.subplots(figsize=(7, 4))
if not plot_df.empty:
    labels = plot_df["region"].astype(str) + "-" + plot_df["condition"].astype(str) + " sweep " + plot_df["sweep"].astype(str)
    x = range(len(plot_df))
    ax.errorbar(x, plot_df["pi_median"], yerr=[plot_df["pi_median"] - plot_df["pi_lower"], plot_df["pi_upper"] - plot_df["pi_median"]], fmt="o")
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("Predicted peak depolarization (mV)")
ax.set_title("Step 06 accepted-ensemble prediction intervals")
fig.tight_layout()

## Feature posterior predictive coverage

In [ ]:
ppc = result["feature_distribution_ppc"]
ppc.sort_values(["region", "condition", "sweep", "feature"]).head(15)

## Perturbation robustness

In [ ]:
perturb = result["perturbation_sweeps"]
perturb_cols = [
    "file_id", "region", "condition", "mechanism_cluster", "buffering_phenotype",
    "current_na", "perturbation", "K_o_peak", "K_o_recovery_error",
    "Vm_feature_pass_fraction", "hidden_flux_plausible", "robust_under_perturbation",
    "perturbation_status", "simulation_status",
]
perturb[perturb_cols].head(30)

In [ ]:
robustness = result["robustness_summary"]
robustness

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
if not robustness.empty:
    robust_plot = robustness.copy()
    robust_plot["label"] = robust_plot["mechanism_cluster"].astype(str) + " / " + robust_plot["region"].astype(str)
    ax.bar(robust_plot["label"], robust_plot["perturbation_robust_fraction"].fillna(0.0))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Robust perturbation fraction")
    ax.tick_params(axis="x", rotation=30)
ax.set_title("Perturbation robustness by mechanism/region")
fig.tight_layout()

## Claim scope after Step 06

The pipeline reports `predictive_supported`, `prediction_limited`, `fit_only`, or `insufficient_evidence`. It also reports a numerical `biological_description_score` that combines held-out, posterior predictive, perturbation, Vm-feature, and hidden-flux support. This score is an evidence-maturity screen, not a degeneracy claim by itself.

The notebook does **not** upgrade clusters to `candidate_degenerate_regimes` from Step 06 alone; any degeneracy wording requires mechanism distinction plus predictive/perturbation support and later assumption-sensitivity and parameter-plausibility checks.

In [ ]:
pd.DataFrame([result["analysis_summary"]])

In [ ]:
assert set(perturb["current_na"].dropna().astype(int)) == {50, 75, 100, 125, 150, 175}
duration = perturb[perturb["perturbation"].astype(str).str.startswith("stimulus_duration")]
assert not duration.empty
assert duration["perturbation_status"].eq("evaluated").all()
assert "biological_description_score" in robustness.columns
assert not robustness["final_biological_degeneracy_claim_allowed"].astype(bool).any()
print("Step 06 full-current validation notebook completed across the full cell-level target scope.")

## Candidate-scope sensitivity

This sensitivity screen compares the default best-per-cell Step 06 validation against top-k and mechanism-diverse accepted representatives per cell. It is a scope-control analysis: it can support robustness of a screen, but final biological degeneracy wording remains gated by the conservative claim logic above.


In [ ]:
from src.step06_predictive_validation import load_step06_inputs

scope_rows = []
scope_robustness_tables = []
for candidate_policy, candidates_per_cell in [
    ("best_per_cell", 1),
    ("top_k_per_cell", 3),
    ("mechanism_diverse_per_cell", 3),
]:
    scope_config = Step06Config(
        max_candidates=None,
        candidate_policy=candidate_policy,
        candidates_per_cell=candidates_per_cell,
        time_points=50,
        perturbation_current_na=None,
        write_outputs=False,
    )
    scope_candidates, _ = load_step06_inputs(PROJECT_ROOT, scope_config)
    scope_result = run_step06_predictive_validation(PROJECT_ROOT, scope_config)
    scope_robustness = scope_result["robustness_summary"].copy()
    scope_robustness.insert(0, "candidate_policy", candidate_policy)
    scope_robustness.insert(1, "candidates_per_cell", candidates_per_cell)
    scope_robustness_tables.append(scope_robustness)
    scope_rows.append({
        "candidate_policy": candidate_policy,
        "candidates_per_cell": candidates_per_cell,
        "n_candidates": int(len(scope_candidates)),
        "n_cells": int(scope_candidates["file_id"].nunique()),
        "n_robustness_rows": int(len(scope_robustness)),
        "n_predictive_supported_rows": int((scope_robustness["validation_label"].astype(str) == "predictive_supported").sum()),
        "n_prediction_limited_rows": int((scope_robustness["validation_label"].astype(str) == "prediction_limited").sum()),
        "mean_biological_description_score": float(pd.to_numeric(scope_robustness["biological_description_score"], errors="coerce").mean()),
        "mean_perturbation_robust_fraction": float(pd.to_numeric(scope_robustness["perturbation_robust_fraction"], errors="coerce").mean()),
    })

scope_sensitivity_summary = pd.DataFrame(scope_rows)
scope_sensitivity_robustness = pd.concat(scope_robustness_tables, ignore_index=True, sort=False)
reviewer_synthesis_dir = PROJECT_ROOT / "outputs" / "reviewer_synthesis"
reviewer_synthesis_dir.mkdir(parents=True, exist_ok=True)
scope_sensitivity_summary.to_csv(reviewer_synthesis_dir / "step06_candidate_scope_sensitivity_summary.csv", index=False)
scope_sensitivity_robustness.to_csv(reviewer_synthesis_dir / "step06_candidate_scope_sensitivity_robustness.csv", index=False)
scope_sensitivity_summary


## Post-execution scientific status

Executed status for reviewer response: Step 06 used one best accepted candidate per represented cell (35 candidates) and evaluated all six canonical currents, producing 280 held-out rows and 1890 perturbation rows. In the canonical best-per-cell screen, 3 mechanism-region-condition groups are `predictive_supported` and 5 remain `prediction_limited`; expanded candidate-scope sensitivity gives 8/13 supported rows for top-3 per cell and 10/15 supported rows for mechanism-diverse top-3 per cell. This strengthens R6 and part of R5 while showing that support improves with candidate diversity, but final biological degeneracy wording remains prohibited because support is not uniform and assumption/parameter checks still constrain the claim.